# NuminaMath-1.5 hard-question set (2,000 questions)

Selects the pipeline's `numina_hard` targets from **AI-MO/NuminaMath-1.5** with the repository's own
`create_numina_hard_question_set` (`src/utils.py`): inside every source block it picks one random question, then
always the question farthest (lowest maximum cosine similarity, BERT embeddings) from the ones already picked.

**Source shares** (aimed at AIME 2026, MATH500 and GSM8K):

| block | source | share |
|---|---|---|
| hard, AIME-like (50%) | amc_aime 20%, cn_contest 10%, aops_forum 10%, olympiads_ref 7%, number_theory 3% | |
| MATH500 / GSM8K-like (35%) | cn_k12 35% | |
| other (15%) | olympiads + inequalities (every unlisted source) | |

**Rows that can be picked**: the corpus filter's usable rows (no synthetic rows, one row per group of copies), only
`problem_is_valid` and `solution_is_valid` = Yes, only math word problems with a final answer (multiple choice only for
amc_aime), at least `MIN_PROBLEM_CHARS` characters, and no copy (also with other numbers) of any MATH500, GSM8K,
AIME 2025 or AIME 2026 question. The farthest search uses the saved NuminaMath-1.5 BERT embeddings over whole pools.

**Output**, uploaded to `mostafabehroozi/numinamath-2025` under `hard_questions/`:
* `hard_questions_n2000_farthest_src-..._seed42_<time>.json`: a plain JSON list of NuminaMath-1.5 row indices (the
  format `HARD_QUESTIONS_INDICES_PATH` reads), at dataset revision `1b05109f`
* `<same name>_report.json`: shares, counts per source, filters, revisions and checks

**Run**: CPU is enough (no accelerator), *Internet on*, Kaggle secret `HF_TOKEN` (write access), *Save & Run All*.
In the pipeline, set `CONFIG["EXEMPLAR_CORPUS"] = "NuminaMath-1.5"` and point `HARD_QUESTIONS_INDICES_PATH` at the file.

In [ ]:
# Settings
from pathlib import Path

REPO_URL = "https://github.com/mostafabehroozi/analogical_math_rag.git"
REPO_BRANCH = "analogical-consistency"
REPO_DIR = Path("/kaggle/working/analogical_math_rag")

TOTAL_QUESTIONS = 2000
SOURCE_PERCENTAGES = {          # every unlisted source forms the "other" pool (15% here: olympiads, inequalities)
    "amc_aime": 20,
    "cn_contest": 10,
    "aops_forum": 10,
    "olympiads_ref": 7,
    "number_theory": 3,
    "cn_k12": 35,
}
SELECTION_STRATEGY = "farthest"
RANDOM_SEED = 42

MIN_PROBLEM_CHARS = 40                          # very short rows are mostly parsing leftovers
MCQ_SOURCES = {"amc_aime"}                      # multiple choice allowed only here (AMC 10/12)
NO_ANSWER = {"", "proof", "notfound", "none", "nan"}
EXCLUDE_BENCHMARK_COPIES = ("math500", "gsm8k", "aime25", "aime26")
SPOT_CHECK_ROWS = 32                            # re-embedded with BERT to prove the embedding file is aligned

TARGET_REPO = "mostafabehroozi/numinamath-2025"
TARGET_DIR_IN_REPO = "hard_questions"
WORK_DIR = Path("/kaggle/working/hard_questions_out")
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Repository code (the exact selector the pipeline uses), Hugging Face token and an early write test
import datetime, json, os, subprocess, sys, time

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
CODE_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True,
                             check=True).stdout.strip()
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print("code commit:", CODE_COMMIT)

from huggingface_hub import HfApi, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("Add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets) and tick it for this notebook.") from exc
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
print("Hugging Face user:", api.whoami()["name"])
STARTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()
(WORK_DIR / "status.json").write_text(json.dumps({"status": "running", "started_at": STARTED_AT}))
api.upload_file(path_or_fileobj=str(WORK_DIR / "status.json"), path_in_repo=f"{TARGET_DIR_IN_REPO}/_run_status.json",
                repo_id=TARGET_REPO, repo_type="dataset", commit_message="Hard-question selection: run started (write test)")
print("Write access confirmed.")

In [ ]:
# NuminaMath-1.5 at the embedded revision, its row order check, and the saved BERT embeddings
import hashlib
import numpy as np
import pandas as pd

from config import CONFIG
from src.corpus_filter import EXEMPLAR_CORPUS_PRESETS, apply_exemplar_corpus, build_corpus_filter
from src.utils import load_exemplar_corpus

CONFIG["EXEMPLAR_CORPUS"] = "NuminaMath-1.5"
CONFIG["CORPUS_FILTER_ENABLED"] = True
apply_exemplar_corpus(CONFIG)
PRESET = EXEMPLAR_CORPUS_PRESETS["NuminaMath-1.5"]
t0 = time.time()
dataset = load_exemplar_corpus(CONFIG)
if dataset is None or len(dataset) != PRESET["rows"]:
    raise RuntimeError(f"NuminaMath-1.5 did not load with {PRESET['rows']} rows.")
meta = dataset.select_columns(
    ["problem", "source", "question_type", "problem_is_valid", "solution_is_valid", "answer"]).to_pandas()
print(f"{len(meta):,} rows loaded in {time.time() - t0:.0f}s")

rows_file = hf_hub_download(PRESET["embedding_source"]["repo_id"], "embeddings/NuminaMath-1.5_rows.parquet",
                            repo_type="dataset", revision=PRESET["embedding_source"]["revision"], token=HF_TOKEN)
sha1 = meta["problem"].map(lambda t: hashlib.sha1((t if isinstance(t, str) else "").encode("utf-8")).hexdigest())
if not (pd.read_parquet(rows_file, columns=["problem_sha1"])["problem_sha1"].to_numpy() == sha1.to_numpy()).all():
    raise RuntimeError("Row order differs from the embedded rows: indices would point at other problems.")
print("Row order matches the embedding file's rows.")

source = PRESET["embedding_source"]
t0 = time.time()
emb_path = hf_hub_download(source["repo_id"], source["filename"], repo_type=source["repo_type"],
                           revision=source["revision"], token=HF_TOKEN)
EMB = np.load(emb_path, mmap_mode="r")
assert EMB.shape == (len(meta), 768) and os.path.getsize(emb_path) == source["bytes"], EMB.shape
print(f"embeddings {EMB.shape} ready in {time.time() - t0:.0f}s")

from sentence_transformers import SentenceTransformer
bert = SentenceTransformer(CONFIG["EMBEDDING_MODEL_PATH_HF"], device="cpu")
probe = np.random.default_rng(0).choice(len(meta), SPOT_CHECK_ROWS, replace=False)
fresh = bert.encode([meta["problem"][i] for i in probe], convert_to_numpy=True)
stored = np.asarray(EMB[np.sort(probe)])[np.argsort(np.argsort(probe))]
cos = (fresh * stored).sum(1) / np.linalg.norm(fresh, axis=1) / np.linalg.norm(stored, axis=1)
SPOT_CHECK = {"rows": int(SPOT_CHECK_ROWS), "cosine_min": float(cos.min())}
print("embedding spot check:", SPOT_CHECK)
assert cos.min() > 0.999, "the embedding file does not match the dataset rows"

In [ ]:
# Which rows may be picked
from src.benchmark_data import load_huggingface_benchmark

corpus_filter = build_corpus_filter(dataset, CONFIG)

answer = meta["answer"].fillna("").astype(str).str.strip().str.lower()
qtype = meta["question_type"].fillna("").astype(str)
src = meta["source"].astype(str).str.strip().str.lower()
valid = (meta["problem_is_valid"] == "Yes") & (meta["solution_is_valid"] == "Yes")
gradable = ((qtype == "math-word-problem") | ((qtype == "MCQ") & src.isin(MCQ_SOURCES))) & ~answer.isin(NO_ANSWER)
long_enough = meta["problem"].fillna("").str.strip().str.len() >= MIN_PROBLEM_CHARS

benchmark_copy = np.zeros(len(meta), dtype=bool)
BENCHMARK_COPIES = {}
for name in EXCLUDE_BENCHMARK_COPIES:
    questions, _ = load_huggingface_benchmark(name)
    hits = [corpus_filter.copies_of(q) for q in questions]
    rows = np.unique(np.concatenate(hits)) if hits else np.zeros(0, dtype=np.int64)
    benchmark_copy[rows] = True
    BENCHMARK_COPIES[name] = {"questions": len(questions), "questions_with_copies": int(sum(h.size > 0 for h in hits)),
                              "corpus_rows": int(rows.size)}
print("benchmark copies:", json.dumps(BENCHMARK_COPIES))

usable = ~corpus_filter.excluded
allowed = (valid & gradable & long_enough).to_numpy() & ~benchmark_copy
ELIGIBILITY = {
    "rows": int(len(meta)), "corpus_filter_usable": int(usable.sum()),
    "not_valid": int((usable & ~valid.to_numpy()).sum()),
    "not_gradable": int((usable & valid.to_numpy() & ~gradable.to_numpy()).sum()),
    "too_short": int((usable & (valid & gradable).to_numpy() & ~long_enough.to_numpy()).sum()),
    "benchmark_copies": int((usable & (valid & gradable & long_enough).to_numpy() & benchmark_copy).sum()),
    "eligible": int((usable & allowed).sum()),
}
print(json.dumps(ELIGIBILITY, indent=1))

pool_table = pd.DataFrame({"source": src, "usable": usable, "eligible": usable & allowed}).groupby("source").sum()
pool_table["rows"] = src.value_counts()
listed = set(SOURCE_PERCENTAGES)
need = {s: TOTAL_QUESTIONS * p / 100 for s, p in SOURCE_PERCENTAGES.items()}
need["other"] = TOTAL_QUESTIONS * (100 - sum(SOURCE_PERCENTAGES.values())) / 100
pool_table["block"] = [s if s in listed else "other" for s in pool_table.index]
print(pool_table[["block", "rows", "usable", "eligible"]].sort_values("eligible", ascending=False).to_string())
print("needed per block:", {k: round(v) for k, v in need.items()})

In [ ]:
# Farthest-point selection inside each source block (repository code)
from src.utils import create_numina_hard_question_set

t0 = time.time()
report = create_numina_hard_question_set(
    dataset=dataset,
    total_questions=TOTAL_QUESTIONS,
    source_percentages=SOURCE_PERCENTAGES,
    selection_strategy=SELECTION_STRATEGY,
    embeddings=EMB,
    output_dir=WORK_DIR,
    random_seed=RANDOM_SEED,
    corpus_filter=corpus_filter,
    allowed_rows=allowed,
)
SELECT_SECONDS = time.time() - t0
selected = report["selected_indices"]
print(f"selected {len(selected)} in {SELECT_SECONDS:.0f}s ->", report["output_path"])
print(json.dumps({k: report[k] for k in ("allocated_counts", "available_counts", "selected_source_counts")}, indent=1))

In [ ]:
# Checks: the saved file, every pick usable/eligible, block counts, and how spread out the set is
saved = json.loads(Path(report["output_path"]).read_text())
assert saved == selected and all(isinstance(i, int) for i in saved), "saved file differs from the selection"
assert len(set(saved)) == TOTAL_QUESTIONS
assert usable[saved].all() and allowed[saved].all(), "a picked row is hidden or not eligible"
blocks = pd.Series([s if s in listed else "other" for s in src[saved]]).value_counts().to_dict()
assert blocks == {k: v for k, v in report["allocated_counts"].items() if v}, (blocks, report["allocated_counts"])


def nearest_neighbour_cosine(rows):
    x = np.asarray(EMB[np.sort(rows)], dtype=np.float32)
    x /= np.linalg.norm(x, axis=1, keepdims=True)
    sim = x @ x.T
    np.fill_diagonal(sim, -1)
    return sim.max(1)


picked_nn = nearest_neighbour_cosine(np.asarray(saved))
random_rows = np.random.default_rng(1).choice(np.flatnonzero(usable & allowed), TOTAL_QUESTIONS, replace=False)
random_nn = nearest_neighbour_cosine(random_rows)
CHECKS = {
    "unique_indices": len(set(saved)), "all_usable_and_eligible": True, "block_counts": blocks,
    "selected_question_types": qtype[saved].value_counts().to_dict(),
    "nearest_neighbour_cosine": {"selected_mean": float(picked_nn.mean()), "selected_max": float(picked_nn.max()),
                                 "random_eligible_mean": float(random_nn.mean())},
    "embedding_spot_check": SPOT_CHECK,
}
print(json.dumps(CHECKS, indent=1))
for block in ["amc_aime", "olympiads_ref", "cn_k12", "other"]:
    rows = [i for i in saved if (src[i] if src[i] in listed else "other") == block][:2]
    for i in rows:
        print(f"\n[{block} | row {i} | {src[i]} | {qtype[i]} | answer {meta['answer'][i]!r}]\n{meta['problem'][i][:400]}")

In [ ]:
# Report, then one commit with the index file and its report
out = Path(report["output_path"])
report_path = out.with_name(out.stem + "_report.json")
full_report = {
    "status": "complete", "started_at": STARTED_AT,
    "finished_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "index_file": out.name,
    "format": "JSON list[int] of NuminaMath-1.5 row indices (HARD_QUESTIONS_INDICES_PATH format)",
    "corpus": "AI-MO/NuminaMath-1.5", "corpus_revision": PRESET["revision"], "rows": int(len(meta)),
    "embeddings": {k: source[k] for k in ("repo_id", "revision", "filename", "sha256")},
    "embedding_model": CONFIG["EMBEDDING_MODEL_PATH_HF"],
    "code": {"repo": REPO_URL, "branch": REPO_BRANCH, "commit": CODE_COMMIT,
             "selector": "src.utils.create_numina_hard_question_set"},
    "settings": {"total_questions": TOTAL_QUESTIONS, "source_percentages": SOURCE_PERCENTAGES,
                 "selection_strategy": SELECTION_STRATEGY, "random_seed": RANDOM_SEED,
                 "min_problem_chars": MIN_PROBLEM_CHARS, "mcq_sources": sorted(MCQ_SOURCES),
                 "no_answer_values": sorted(NO_ANSWER), "excluded_benchmark_copies": list(EXCLUDE_BENCHMARK_COPIES),
                 "corpus_filter": corpus_filter.signature},
    "eligibility": ELIGIBILITY, "benchmark_copies": BENCHMARK_COPIES,
    "selection": {k: v for k, v in report.items() if k not in ("selected_indices", "output_path")},
    "select_seconds": round(SELECT_SECONDS, 1), "checks": CHECKS,
}
report_path.write_text(json.dumps(full_report, indent=2, default=str))
print(json.dumps({k: full_report[k] for k in ("index_file", "settings", "eligibility")}, indent=1, default=str))

upload_dir = WORK_DIR / "upload"
upload_dir.mkdir(exist_ok=True)
for path in (out, report_path):
    (upload_dir / path.name).write_bytes(path.read_bytes())
api.upload_folder(folder_path=str(upload_dir), path_in_repo=TARGET_DIR_IN_REPO, repo_id=TARGET_REPO,
                  repo_type="dataset", commit_message=f"NuminaMath-1.5 hard-question set: {out.name}")
api.delete_file(f"{TARGET_DIR_IN_REPO}/_run_status.json", repo_id=TARGET_REPO, repo_type="dataset",
                commit_message="Hard-question selection: remove the run-status file")
remote = {f.path: getattr(f, "size", None)
          for f in api.list_repo_tree(TARGET_REPO, path_in_repo=TARGET_DIR_IN_REPO, repo_type="dataset")}
print(json.dumps(remote, indent=1))
assert remote.get(f"{TARGET_DIR_IN_REPO}/{out.name}") == out.stat().st_size
print(f"\nDONE: https://huggingface.co/datasets/{TARGET_REPO}/tree/main/{TARGET_DIR_IN_REPO}")